# 🏭 CASO 3 — PRODUCCIÓN INDUSTRIAL
## Visualización de Datos con Streamlit
### 📋 TAREA — Desarrollo desde cero

---

## Contexto del problema

**Empresa:** MetalParts Colombia S.A.S.  
**Sector:** Manufactura de piezas metálicas (tornillería y fijaciones)  
**Sede:** Zona Industrial de Itagüí, Antioquia

MetalParts opera 4 líneas de producción con diferentes máquinas y turnos. La gerencia de operaciones solicita un **dashboard de control de producción** para tomar decisiones basadas en datos. Actualmente los reportes se hacen en Excel manualmente y toman 2 días en prepararse.

**El gerente necesita responder estas preguntas:**
1. ¿Cuál es la eficiencia promedio por línea de producción?
2. ¿En qué turno se producen más defectos?
3. ¿Qué máquina genera más tiempos de paro?
4. ¿Cómo ha evolucionado la producción semana a semana?
5. ¿Cuál es la relación entre temperatura y tasa de defectos?

---

## 📌 Tu entregable

Debes construir:
1. **Este notebook** con el análisis exploratorio completo
2. **Un archivo `caso3_produccion_app.py`** con la app Streamlit funcional
3. **Captura de pantalla** de la app corriendo en tu computador

---

## 📊 Dataset disponible

Archivo: `caso3_produccion_dataset.csv`

| Columna | Tipo | Descripción |
|---|---|---|
| id_orden | str | Identificador de orden de producción |
| fecha_produccion | date | Fecha de la orden |
| linea_produccion | str | Línea A, B, C o D |
| producto | str | Nombre del producto fabricado |
| turno | str | Mañana, Tarde o Noche |
| operador | str | Código del operador |
| maquina | str | Código de la máquina |
| unidades_planificadas | int | Meta de producción |
| unidades_producidas | int | Producción real |
| unidades_defectuosas | int | Unidades con defecto |
| tiempo_ciclo_min | float | Tiempo por unidad (minutos) |
| tiempo_paro_min | float | Minutos de paro no planificado |
| causa_paro | str | Motivo del paro |
| temperatura_c | float | Temperatura del proceso |
| consumo_energia_kwh | float | Consumo energético |
| costo_produccion_cop | float | Costo total de la orden |
| eficiencia_pct | float | % de eficiencia (producido/planificado) |
| tasa_defectos_pct | float | % de defectos sobre producidos |
| semana | int | Número de semana del año |

---

## 🎯 Requisitos mínimos

### En el notebook:
- [ ] Cargar y explorar el dataset (shape, tipos, describe)
- [ ] Calcular al menos 4 KPIs del negocio
- [ ] Crear mínimo 5 visualizaciones Plotly respondiendo las preguntas del gerente
- [ ] Cada gráfica debe tener título y etiquetas claras
- [ ] Identificar al menos 1 insight o hallazgo relevante

### En la app Streamlit:
- [ ] Usar `st.set_page_config()` con título y layout wide
- [ ] Mínimo 2 filtros en el sidebar
- [ ] Mínimo 4 KPIs con `st.metric()`
- [ ] Mínimo 4 gráficas Plotly integradas con `st.plotly_chart()`
- [ ] Usar `st.columns()` para el layout (patrón F o Z)
- [ ] Usar `@st.cache_data` para la carga de datos

---

## 🏆 Puntos extra (opcional)
- Agregar un **selector de rango de fechas** con `st.date_input()`
- Mostrar una tabla de **órdenes con defectos > 10%** como alerta
- Usar `st.tabs()` para organizar las secciones del dashboard
- Agregar `st.download_button()` para exportar el dataset filtrado

---

## ⏰ Tiempo estimado: 45–60 minutos

---

# 🚀 ¡Empieza aquí!

In [6]:
# ══════════════════════════════════════════════════════════════
# PASO 1 — Importar librerías
# ══════════════════════════════════════════════════════════════

# Importamos las librerías necesarias para:
# - Manipulación de datos → pandas
# - Operaciones numéricas → numpy
# - Visualizaciones interactivas → plotly
# - Construcción de la app → streamlit

import streamlit as st
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.graph_objects as go

print('✅ Librerías importadas correctamente')

✅ Librerías importadas correctamente


In [7]:
# ══════════════════════════════════════════════════════════════
# PASO 2 — Cargar y explorar el dataset
# Archivo: 'caso3_produccion_dataset.csv'
# ══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
# pd.read_csv() carga el archivo CSV y lo convierte en un
# DataFrame de pandas (estructura tipo tabla de Excel)
# ─────────────────────────────────────────────────────────────
df = pd.read_csv('caso3_produccion_dataset.csv')

# ─────────────────────────────────────────────────────────────
# Convertimos la columna de fecha a tipo datetime
# para poder filtrar, agrupar y analizar por fechas
# ─────────────────────────────────────────────────────────────
df['fecha_produccion'] = pd.to_datetime(df['fecha_produccion'])

# ─────────────────────────────────────────────────────────────
# Exploración básica del dataset
# ─────────────────────────────────────────────────────────────

# .shape devuelve: (filas, columnas)
print(f'📋 Número de filas    : {df.shape[0]}')
print(f'📋 Número de columnas : {df.shape[1]}')

# Mostrar primeras filas
print('\n🔍 Primeras 5 filas del dataset:')
display(df.head())

# ─────────────────────────────────────────────────────────────
# Información general del dataset:
# - Tipos de datos
# - Valores no nulos
# - Uso de memoria
# ─────────────────────────────────────────────────────────────
print('\n📊 Información general:')
df.info()

# ─────────────────────────────────────────────────────────────
# Estadísticas descriptivas de columnas numéricas
# mean = promedio
# std  = desviación estándar
# min/max = valores extremos
# ─────────────────────────────────────────────────────────────
print('\n📈 Estadísticas descriptivas:')
display(df.describe().round(2))

# ─────────────────────────────────────────────────────────────
# Explorar columnas categóricas
# value_counts() cuenta cuántas veces aparece cada categoría
# ─────────────────────────────────────────────────────────────
columnas_cat = [
    'linea_produccion',
    'producto',
    'turno',
    'maquina',
    'causa_paro'
]

for col in columnas_cat:
    print(f'\n🏷️ Valores en {col}:')
    print(df[col].value_counts().to_string())

📋 Número de filas    : 160
📋 Número de columnas : 19

🔍 Primeras 5 filas del dataset:


,id_orden,fecha_produccion,linea_produccion,producto,turno,operador,maquina,unidades_planificadas,unidades_producidas,unidades_defectuosas,tiempo_ciclo_min,tiempo_paro_min,causa_paro,temperatura_c,consumo_energia_kwh,costo_produccion_cop,eficiencia_pct,tasa_defectos_pct,semana
0,OP-2000,2024-06-14,Línea D,Remache 6mm,Tarde,OP-106,Torno-02,2235,1443,91,3.76,30.2,Sin causa,23.6,528.88,2147000.0,64.56,6.31,24
1,OP-2001,2024-03-18,Línea A,Tuerca M8,Tarde,OP-109,CNC-02,3024,4317,66,5.33,61.8,Sin causa,25.0,439.95,4480000.0,100.00,1.53,12
2,OP-2002,2024-07-21,Línea B,Tuerca M8,Tarde,OP-109,CNC-01,3293,4278,103,7.70,114.0,Falla eléctrica,38.0,783.20,1584000.0,100.00,2.41,29
3,OP-2003,2024-11-29,Línea C,Varilla Roscada,Noche,OP-109,CNC-02,2839,2150,87,2.52,103.3,Mantenimiento,18.6,243.94,815000.0,75.73,4.05,48
4,OP-2004,2024-01-25,Línea D,Perno M12,Noche,OP-104,CNC-01,3741,1445,62,3.45,39.0,Falla eléctrica,43.8,579.14,272000.0,38.63,4.29,4



📊 Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_orden               160 non-null    object        
 1   fecha_produccion       160 non-null    datetime64[ns]
 2   linea_produccion       160 non-null    object        
 3   producto               160 non-null    object        
 4   turno                  160 non-null    object        
 5   operador               160 non-null    object        
 6   maquina                160 non-null    object        
 7   unidades_planificadas  160 non-null    int64         
 8   unidades_producidas    160 non-null    int64         
 9   unidades_defectuosas   160 non-null    int64         
 10  tiempo_ciclo_min       160 non-null    float64       
 11  tiempo_paro_min        160 non-null    float64       
 12  causa_paro             160 non-null    o

,fecha_produccion,unidades_planificadas,unidades_producidas,unidades_defectuosas,tiempo_ciclo_min,tiempo_paro_min,temperatura_c,consumo_energia_kwh,costo_produccion_cop,eficiencia_pct,tasa_defectos_pct,semana
count,160,160.00,160.00,160.00,160.00,160.00,160.00,160.00,160.00,160.00,160.00,160.00
mean,2024-07-02 18:09:00,2804.63,2591.72,90.21,4.20,56.88,30.68,432.34,2477387.50,76.72,5.52,26.50
min,2024-01-12 00:00:00,550.00,438.00,0.00,0.53,0.70,18.20,55.33,207000.00,10.99,0.00,1.00
25%,2024-03-23 06:00:00,1941.25,1448.00,47.75,2.44,25.70,23.70,236.52,1376000.00,53.12,1.63,12.00
50%,2024-07-07 12:00:00,2826.00,2527.50,89.00,4.08,52.80,30.10,438.59,2266500.00,91.66,3.39,27.00
75%,2024-10-08 06:00:00,3824.50,3651.50,133.25,6.06,87.18,37.12,600.36,3562750.00,100.00,6.54,41.00
max,2024-12-30 00:00:00,4794.00,4932.00,199.00,7.98,118.00,44.90,798.39,4990000.00,100.00,33.40,52.00
std,NaN,1166.51,1294.43,53.34,2.08,34.70,7.89,220.77,1364353.09,28.69,6.19,15.12



🏷️ Valores en linea_produccion:
linea_produccion
Línea D    56
Línea A    42
Línea B    33
Línea C    29

🏷️ Valores en producto:
producto
Tuerca M8          24
Remache 6mm        23
Tornillo M8        22
Varilla Roscada    21
Alambre Galv.      19
Perno M12          18
Clavo 3"           18
Arandela 10mm      15

🏷️ Valores en turno:
turno
Noche     56
Tarde     52
Mañana    52

🏷️ Valores en maquina:
maquina
Torno-02     34
CNC-01       27
Prensa-02    27
Torno-01     26
Prensa-01    24
CNC-02       22

🏷️ Valores en causa_paro:
causa_paro
Sin causa          72
Falla eléctrica    31
Mantenimiento      30
Cambio de turno    27


In [8]:
# ══════════════════════════════════════════════════════════════
# PASO 3 — Calcular KPIs
# Indicadores clave de desempeño (KPIs)
# ══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
# KPI 1 — Eficiencia promedio general
#
# mean() calcula el promedio de una columna numérica.
# La eficiencia representa qué porcentaje de la meta
# planificada realmente fue producido.
# ─────────────────────────────────────────────────────────────
eficiencia_promedio = df['eficiencia_pct'].mean()

# ─────────────────────────────────────────────────────────────
# KPI 2 — Tasa promedio de defectos
#
# Indica qué porcentaje de unidades producidas
# presentó defectos de calidad.
# Mientras menor sea este valor, mejor.
# ─────────────────────────────────────────────────────────────
tasa_defectos_prom = df['tasa_defectos_pct'].mean()

# ─────────────────────────────────────────────────────────────
# KPI 3 — Total de unidades producidas
#
# sum() acumula todas las unidades fabricadas.
# Permite medir el volumen total de producción.
# ─────────────────────────────────────────────────────────────
total_unidades = df['unidades_producidas'].sum()

# ─────────────────────────────────────────────────────────────
# KPI 4 — Costo total de producción
#
# Representa cuánto dinero costó fabricar todas
# las órdenes registradas en el dataset.
# ─────────────────────────────────────────────────────────────
costo_total = df['costo_produccion_cop'].sum()

# ─────────────────────────────────────────────────────────────
# KPI 5 — Tiempo total de paro
#
# Los tiempos de paro representan minutos donde
# las máquinas no estuvieron produciendo.
# Es uno de los indicadores más importantes en
# manufactura industrial.
# ─────────────────────────────────────────────────────────────
tiempo_paro_total = df['tiempo_paro_min'].sum()

# ─────────────────────────────────────────────────────────────
# KPI 6 — Total de unidades defectuosas
#
# Permite dimensionar el impacto total de problemas
# de calidad en la producción.
# ─────────────────────────────────────────────────────────────
unidades_defectuosas = df['unidades_defectuosas'].sum()

# ─────────────────────────────────────────────────────────────
# KPI 7 — Producción promedio por orden
#
# Mide cuántas unidades produce en promedio cada
# orden de producción.
# ─────────────────────────────────────────────────────────────
produccion_promedio = df['unidades_producidas'].mean()

# ─────────────────────────────────────────────────────────────
# KPI 8 — Consumo total de energía
#
# Indicador importante para monitorear eficiencia
# energética y costos operativos.
# ─────────────────────────────────────────────────────────────
energia_total = df['consumo_energia_kwh'].sum()

# ─────────────────────────────────────────────────────────────
# KPI 9 — Máquina con mayor tiempo de paro
#
# groupby() agrupa por máquina
# sum() acumula minutos de paro
# idxmax() devuelve la fila con el valor máximo
# ─────────────────────────────────────────────────────────────
paro_maquinas = (
    df.groupby('maquina')['tiempo_paro_min']
      .sum()
      .reset_index()
)

maquina_mas_paros = paro_maquinas.loc[
    paro_maquinas['tiempo_paro_min'].idxmax()
]

# ─────────────────────────────────────────────────────────────
# KPI 10 — Línea más eficiente
#
# Calculamos el promedio de eficiencia por línea
# para identificar la mejor línea de producción.
# ─────────────────────────────────────────────────────────────
eficiencia_lineas = (
    df.groupby('linea_produccion')['eficiencia_pct']
      .mean()
      .reset_index()
)

linea_mas_eficiente = eficiencia_lineas.loc[
    eficiencia_lineas['eficiencia_pct'].idxmax()
]

# ══════════════════════════════════════════════════════════════
# Mostrar resultados
# ══════════════════════════════════════════════════════════════

print('📊 KPIs GENERALES DE PRODUCCIÓN')
print('─' * 55)

print(f'⚙️ Eficiencia promedio general : {eficiencia_promedio:.2f}%')
print(f'❌ Tasa promedio de defectos   : {tasa_defectos_prom:.2f}%')
print(f'🏭 Total unidades producidas   : {total_unidades:,}')
print(f'💰 Costo total producción      : ${costo_total:,.0f} COP')
print(f'⏱️ Tiempo total de paro        : {tiempo_paro_total:,.1f} min')
print(f'🛑 Unidades defectuosas        : {unidades_defectuosas:,}')
print(f'📦 Producción promedio/orden   : {produccion_promedio:,.0f} unidades')
print(f'⚡ Consumo total energía       : {energia_total:,.2f} kWh')

print('\n🏆 HALLAZGOS OPERACIONALES')
print('─' * 55)

print(
    f'🔧 Máquina con más paros       : '
    f'{maquina_mas_paros["maquina"]} '
    f'({maquina_mas_paros["tiempo_paro_min"]:.1f} min)'
)

print(
    f'🚀 Línea más eficiente         : '
    f'{linea_mas_eficiente["linea_produccion"]} '
    f'({linea_mas_eficiente["eficiencia_pct"]:.2f}%)'
)

📊 KPIs GENERALES DE PRODUCCIÓN
───────────────────────────────────────────────────────
⚙️ Eficiencia promedio general : 76.72%
❌ Tasa promedio de defectos   : 5.52%
🏭 Total unidades producidas   : 414,675
💰 Costo total producción      : $396,382,000 COP
⏱️ Tiempo total de paro        : 9,100.5 min
🛑 Unidades defectuosas        : 14,434
📦 Producción promedio/orden   : 2,592 unidades
⚡ Consumo total energía       : 69,174.09 kWh

🏆 HALLAZGOS OPERACIONALES
───────────────────────────────────────────────────────
🔧 Máquina con más paros       : Torno-02 (2115.7 min)
🚀 Línea más eficiente         : Línea C (80.20%)


In [9]:
# ══════════════════════════════════════════════════════════════
# PASO 4 — Análisis por grupos
# Agrupaciones y métricas operacionales
# ══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
# groupby() permite agrupar registros por una categoría
# específica y calcular métricas sobre cada grupo.
#
# agg() aplica funciones de agregación:
#   mean  → promedio
#   sum   → suma total
#   count → conteo
#   max   → valor máximo
# ─────────────────────────────────────────────────────────────

# ══════════════════════════════════════════════════════════════
# ANÁLISIS 1 — Eficiencia por línea de producción
# ══════════════════════════════════════════════════════════════

# Calculamos:
# - eficiencia promedio
# - producción total
# - defectos promedio
# por cada línea de producción

eficiencia_linea = (
    df.groupby('linea_produccion')
      .agg(
          eficiencia_promedio = ('eficiencia_pct', 'mean'),
          total_producido     = ('unidades_producidas', 'sum'),
          defectos_promedio   = ('tasa_defectos_pct', 'mean'),
          tiempo_paro_total   = ('tiempo_paro_min', 'sum')
      )
      .round(2)
      .reset_index()
      .sort_values('eficiencia_promedio', ascending=False)
)

print('⚙️ Eficiencia por línea de producción:')
display(eficiencia_linea)

# ══════════════════════════════════════════════════════════════
# ANÁLISIS 2 — Defectos por turno
# ══════════════════════════════════════════════════════════════

# Analizamos qué turno presenta:
# - más defectos
# - peor tasa de calidad
# - más unidades defectuosas

defectos_turno = (
    df.groupby('turno')
      .agg(
          defectos_totales   = ('unidades_defectuosas', 'sum'),
          tasa_defectos_prom = ('tasa_defectos_pct', 'mean'),
          produccion_total   = ('unidades_producidas', 'sum')
      )
      .round(2)
      .reset_index()
      .sort_values('tasa_defectos_prom', ascending=False)
)

print('❌ Defectos por turno:')
display(defectos_turno)

# ══════════════════════════════════════════════════════════════
# ANÁLISIS 3 — Tiempo de paro por máquina
# ══════════════════════════════════════════════════════════════

# Este análisis permite identificar cuáles máquinas
# generan más interrupciones operativas.

paro_maquina = (
    df.groupby('maquina')
      .agg(
          tiempo_paro_total = ('tiempo_paro_min', 'sum'),
          paro_promedio     = ('tiempo_paro_min', 'mean'),
          ordenes           = ('id_orden', 'count')
      )
      .round(2)
      .reset_index()
      .sort_values('tiempo_paro_total', ascending=False)
)

print('⏱️ Tiempo de paro por máquina:')
display(paro_maquina)

# ══════════════════════════════════════════════════════════════
# ANÁLISIS 4 — Producción semanal
# ══════════════════════════════════════════════════════════════

# Agrupamos por número de semana para analizar
# cómo evoluciona la producción en el tiempo.

produccion_semanal = (
    df.groupby('semana')
      .agg(
          unidades_producidas = ('unidades_producidas', 'sum'),
          eficiencia_promedio = ('eficiencia_pct', 'mean'),
          defectos_totales    = ('unidades_defectuosas', 'sum')
      )
      .round(2)
      .reset_index()
)

print('📅 Producción semanal:')
display(produccion_semanal)

# ══════════════════════════════════════════════════════════════
# ANÁLISIS 5 — Temperatura y calidad
# ══════════════════════════════════════════════════════════════

# Exploramos si existe relación entre:
# - temperatura del proceso
# - defectos de producción

temp_defectos = (
    df.groupby('linea_produccion')
      .agg(
          temperatura_promedio = ('temperatura_c', 'mean'),
          defectos_promedio    = ('tasa_defectos_pct', 'mean')
      )
      .round(2)
      .reset_index()
)

print('🌡️ Temperatura vs tasa de defectos:')
display(temp_defectos)

# ══════════════════════════════════════════════════════════════
# ANÁLISIS 6 — Causas de paro más frecuentes
# ══════════════════════════════════════════════════════════════

# value_counts() cuenta cuántas veces aparece cada causa

causas_paro = (
    df['causa_paro']
      .value_counts()
      .reset_index()
)

causas_paro.columns = ['causa_paro', 'cantidad']

print('🛑 Frecuencia de causas de paro:')
display(causas_paro)

# ══════════════════════════════════════════════════════════════
# HALLAZGOS PRINCIPALES
# ══════════════════════════════════════════════════════════════

print('📌 INSIGHTS IDENTIFICADOS')
print('─' * 60)

# Línea más eficiente
mejor_linea = eficiencia_linea.iloc[0]

print(
    f'🚀 La línea más eficiente es {mejor_linea["linea_produccion"]} '
    f'con {mejor_linea["eficiencia_promedio"]:.2f}% de eficiencia promedio.'
)

# Turno con más defectos
peor_turno = defectos_turno.iloc[0]

print(
    f'⚠️ El turno con mayor tasa de defectos es {peor_turno["turno"]} '
    f'con {peor_turno["tasa_defectos_prom"]:.2f}% promedio.'
)

# Máquina con más paros
maquina_critica = paro_maquina.iloc[0]

print(
    f'🔧 La máquina con más tiempo acumulado de paro es '
    f'{maquina_critica["maquina"]} '
    f'({maquina_critica["tiempo_paro_total"]:.1f} minutos).'
)

# Causa más frecuente
causa_principal = causas_paro.iloc[0]

print(
    f'🛑 La causa de paro más frecuente es '
    f'"{causa_principal["causa_paro"]}" '
    f'con {causa_principal["cantidad"]} registros.'
)

⚙️ Eficiencia por línea de producción:


,linea_produccion,eficiencia_promedio,total_producido,defectos_promedio,tiempo_paro_total
2,Línea C,80.20,82583,4.48,1351.5
1,Línea B,79.23,86027,4.21,2211.8
0,Línea A,78.44,109195,5.55,2754.9
3,Línea D,72.15,136870,6.81,2782.3


❌ Defectos por turno:


,turno,defectos_totales,tasa_defectos_prom,produccion_total
0,Mañana,4750,6.18,123672
2,Tarde,4724,5.61,136588
1,Noche,4960,4.82,154415


⏱️ Tiempo de paro por máquina:


...

🌡️ Temperatura vs tasa de defectos:


,linea_produccion,temperatura_promedio,defectos_promedio
0,Línea A,30.20,5.55
1,Línea B,32.38,4.21
2,Línea C,30.50,4.48
3,Línea D,30.13,6.81


🛑 Frecuencia de causas de paro:


,causa_paro,cantidad
0,Sin causa,72
1,Falla eléctrica,31
2,Mantenimiento,30
3,Cambio de turno,27


📌 INSIGHTS IDENTIFICADOS
────────────────────────────────────────────────────────────
🚀 La línea más eficiente es Línea C con 80.20% de eficiencia promedio.
⚠️ El turno con mayor tasa de defectos es Mañana con 6.18% promedio.
🔧 La máquina con más tiempo acumulado de paro es Torno-02 (2115.7 minutos).
🛑 La causa de paro más frecuente es "Sin causa" con 72 registros.


In [10]:
# ══════════════════════════════════════════════════════════════
# PASO 5 — Visualización 1
# Pregunta: ¿Cuál es la eficiencia por línea de producción?
# Tipo: Gráfico de barras
# ══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
# px.bar() crea gráficos de barras interactivos.
#
# x → categorías
# y → valores numéricos
# color → colorea según una variable
# text_auto → muestra etiquetas automáticas
# ─────────────────────────────────────────────────────────────

fig1 = px.bar(
    eficiencia_linea,
    x='linea_produccion',
    y='eficiencia_promedio',
    color='eficiencia_promedio',
    text_auto='.2f',
    title='Eficiencia Promedio por Línea de Producción',
    labels={
        'linea_produccion': 'Línea de Producción',
        'eficiencia_promedio': 'Eficiencia (%)'
    },
    color_continuous_scale='Viridis'
)

# ─────────────────────────────────────────────────────────────
# update_layout() personaliza el diseño general
# ─────────────────────────────────────────────────────────────
fig1.update_layout(
    title_font_size=18,
    xaxis_title='Línea',
    yaxis_title='Eficiencia Promedio (%)',
    showlegend=False
)

# ─────────────────────────────────────────────────────────────
# update_traces() modifica propiedades visuales
# ─────────────────────────────────────────────────────────────
fig1.update_traces(
    textposition='outside'
)

# Mostrar gráfica
fig1.show()

# ══════════════════════════════════════════════════════════════
# INSIGHT
# ══════════════════════════════════════════════════════════════

print('📌 Insight:')
print(
    'La gráfica permite identificar cuáles líneas de producción '
    'presentan mejor desempeño operativo y cuáles requieren '
    'mejoras en eficiencia.'
)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 6 — Visualización 2
# Pregunta: ¿En qué turno se producen más defectos?
# Tipo: Barras agrupadas
# ══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
# Usaremos el DataFrame "defectos_turno" creado anteriormente.
#
# Queremos comparar:
# - defectos totales
# - tasa promedio de defectos
# entre los distintos turnos.
# ─────────────────────────────────────────────────────────────

fig2 = px.bar(
    defectos_turno,
    x='turno',
    y='tasa_defectos_prom',
    color='turno',
    text_auto='.2f',
    title='Tasa Promedio de Defectos por Turno',
    labels={
        'turno': 'Turno de Producción',
        'tasa_defectos_prom': 'Tasa de Defectos (%)'
    },

    color_discrete_sequence = [
        '#440154',  # morado oscuro
        '#21918c',  # verde azulado
        '#fde725'   # amarillo
    ]
)

# ─────────────────────────────────────────────────────────────
# Personalización del layout
# ─────────────────────────────────────────────────────────────
fig2.update_layout(
    title_font_size=18,
    xaxis_title='Turno',
    yaxis_title='Tasa de Defectos (%)',
    showlegend=False
)

# ─────────────────────────────────────────────────────────────
# Personalización de etiquetas
# ─────────────────────────────────────────────────────────────
fig2.update_traces(
    textposition='outside'
)

# Mostrar gráfica
fig2.show()

# ══════════════════════════════════════════════════════════════
# VISUALIZACIÓN EXTRA — Distribución de defectos
# Tipo: Violín
# ══════════════════════════════════════════════════════════════

# Un gráfico de violín muestra:
# - distribución de datos
# - densidad
# - dispersión
# - posibles valores extremos
#
# Es útil para comparar variabilidad entre turnos.

fig2_extra = px.violin(
    df,
    x='turno',
    y='tasa_defectos_pct',
    color='turno',
    box=True,          # agrega boxplot interno
    points='all',      # muestra todos los puntos
    title='Distribución de Tasa de Defectos por Turno',
    labels={
        'turno': 'Turno',
        'tasa_defectos_pct': 'Tasa de Defectos (%)'
    },
    color_discrete_sequence = [
        '#440154',  # morado oscuro
        '#21918c',  # verde azulado
        '#fde725'   # amarillo
    ]
)

fig2_extra.update_layout(
    title_font_size=18
)

fig2_extra.show()

# ══════════════════════════════════════════════════════════════
# INSIGHT
# ══════════════════════════════════════════════════════════════

turno_critico = defectos_turno.iloc[0]

print('📌 Insight:')
print(
    f'El turno con mayor tasa promedio de defectos es '
    f'{turno_critico["turno"]}, lo que podría indicar '
    f'problemas relacionados con fatiga operativa, '
    f'calibración de maquinaria o supervisión.'
)

📌 Insight:
El turno con mayor tasa promedio de defectos es Mañana, lo que podría indicar problemas relacionados con fatiga operativa, calibración de maquinaria o supervisión.


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 7 — Visualización 3
# Pregunta: ¿Qué máquina genera más tiempos de paro?
# Tipo: Barras horizontales
# ══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
# Usaremos el DataFrame "paro_maquina" creado anteriormente.
#
# Las barras horizontales son ideales cuando:
# - las etiquetas son largas
# - queremos comparar rankings
# ─────────────────────────────────────────────────────────────

fig3 = px.bar(
    paro_maquina.sort_values('tiempo_paro_total', ascending=True),

    x='tiempo_paro_total',
    y='maquina',

    orientation='h',   # h = horizontal

    color='tiempo_paro_total',

    text_auto='.1f',

    title='Tiempo Total de Paro por Máquina',

    labels={
        'maquina': 'Máquina',
        'tiempo_paro_total': 'Tiempo de Paro (minutos)'
    },

    # Paleta estilo Viridis
    color_continuous_scale='Viridis'
)

# ─────────────────────────────────────────────────────────────
# Personalización del diseño
# ─────────────────────────────────────────────────────────────
fig3.update_layout(
    title_font_size=18,
    xaxis_title='Tiempo Total de Paro (min)',
    yaxis_title='Máquina',
    height=450,
    showlegend=False
)

# ─────────────────────────────────────────────────────────────
# Personalización de texto
# ─────────────────────────────────────────────────────────────
fig3.update_traces(
    textposition='outside'
)

# Mostrar gráfica
fig3.show()

# ══════════════════════════════════════════════════════════════
# INSIGHT
# ══════════════════════════════════════════════════════════════

maquina_critica = paro_maquina.iloc[0]

print('📌 Insight:')
print(
    f'La máquina {maquina_critica["maquina"]} registra el mayor '
    f'tiempo acumulado de paro, lo que puede impactar '
    f'directamente la productividad y los costos operativos.'
)

📌 Insight:
La máquina Torno-02 registra el mayor tiempo acumulado de paro, lo que puede impactar directamente la productividad y los costos operativos.


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 8 — Visualización 4
# Pregunta: ¿Cómo evoluciona la producción semanal?
# Tipo: Línea de tiempo
# ══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
# px.line() se usa para visualizar evolución temporal.
#
# markers=True agrega puntos visibles en cada semana.
# Esto mejora la lectura de tendencias.
# ─────────────────────────────────────────────────────────────

fig4 = px.line(
    produccion_semanal,

    x='semana',
    y='unidades_producidas',

    markers=True,

    title='Evolución de la Producción por Semana',

    labels={
        'semana': 'Semana del Año',
        'unidades_producidas': 'Unidades Producidas'
    }
)

# ─────────────────────────────────────────────────────────────
# Personalización visual de línea y marcadores
# ─────────────────────────────────────────────────────────────
fig4.update_traces(
    line_color='#21918c',   # color estilo Viridis
    line_width=4,
    marker_size=9
)

# ─────────────────────────────────────────────────────────────
# Personalización del layout
# ─────────────────────────────────────────────────────────────
fig4.update_layout(
    title_font_size=18,
    xaxis_title='Semana',
    yaxis_title='Producción Total',
    hovermode='x unified'
)

# Mostrar gráfica
fig4.show()

# ══════════════════════════════════════════════════════════════
# VISUALIZACIÓN EXTRA — Eficiencia semanal
# ══════════════════════════════════════════════════════════════

# Podemos analizar también cómo cambia la eficiencia
# promedio a lo largo del tiempo.

fig4_extra = px.line(
    produccion_semanal,

    x='semana',
    y='eficiencia_promedio',

    markers=True,

    title='Evolución de la Eficiencia Promedio por Semana',

    labels={
        'semana': 'Semana',
        'eficiencia_promedio': 'Eficiencia (%)'
    }
)

fig4_extra.update_traces(
    line_color='#440154',
    line_width=4,
    marker_size=8
)

fig4_extra.update_layout(
    title_font_size=18,
    hovermode='x unified'
)

fig4_extra.show()

# ══════════════════════════════════════════════════════════════
# INSIGHT
# ══════════════════════════════════════════════════════════════

semana_max = produccion_semanal.loc[
    produccion_semanal['unidades_producidas'].idxmax()
]

print('📌 Insight:')
print(
    f'La semana con mayor volumen de producción fue la '
    f'semana {int(semana_max["semana"])}, alcanzando '
    f'{int(semana_max["unidades_producidas"]):,} unidades producidas.'
)

📌 Insight:
La semana con mayor volumen de producción fue la semana 43, alcanzando 28,856 unidades producidas.


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 9 — Visualización 5
# Pregunta: ¿Existe relación entre temperatura y defectos?
# Tipo: Scatter plot con línea de tendencia
# ══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
# px.scatter() crea diagramas de dispersión.
#
# Cada punto representa una orden de producción.
#
# x → temperatura
# y → tasa de defectos
#
# trendline='ols'
# agrega una línea de regresión lineal para identificar
# posibles correlaciones entre variables.
# ─────────────────────────────────────────────────────────────

fig5 = px.scatter(
    df,

    x='temperatura_c',
    y='tasa_defectos_pct',

    color='linea_produccion',
    size='unidades_producidas',

    hover_data=[
        'maquina',
        'turno',
        'producto'
    ],

    trendline='ols',

    title='Relación entre Temperatura y Tasa de Defectos',

    labels={
        'temperatura_c': 'Temperatura (°C)',
        'tasa_defectos_pct': 'Tasa de Defectos (%)',
        'linea_produccion': 'Línea de Producción'
    },

    # Paleta estilo Viridis
    color_discrete_sequence=[
        '#440154',
        '#31688e',
        '#35b779',
        '#fde725'
    ]
)

# ─────────────────────────────────────────────────────────────
# Personalización del layout
# ─────────────────────────────────────────────────────────────
fig5.update_layout(
    title_font_size=18,
    xaxis_title='Temperatura del Proceso (°C)',
    yaxis_title='Tasa de Defectos (%)',
    legend_title='Línea',
    height=600
)

# ─────────────────────────────────────────────────────────────
# Mejorar visualización de puntos
# ─────────────────────────────────────────────────────────────
fig5.update_traces(
    marker=dict(
        line=dict(width=1, color='DarkSlateGrey')
    )
)

# Mostrar gráfica
fig5.show()

# ══════════════════════════════════════════════════════════════
# INSIGHT
# ══════════════════════════════════════════════════════════════

correlacion = df['temperatura_c'].corr(df['tasa_defectos_pct'])

print('📌 Insight:')
print(
    f'La correlación entre temperatura y tasa de defectos es '
    f'{correlacion:.2f}. '
    f'Esto permite evaluar si incrementos de temperatura '
    f'podrían estar asociados con problemas de calidad.'
)

📌 Insight:
La correlación entre temperatura y tasa de defectos es 0.12. Esto permite evaluar si incrementos de temperatura podrían estar asociados con problemas de calidad.


## 💡 Espacio para Insights

1. ¿Cuál es la eficiencia promedio por línea de producción?
   El análisis muestra una eficiencia promedio general de **76.72%**, aunque existe variabilidad significativa entre líneas. La **Línea C** alcanzó el mejor desempeño con **80.20%**, mientras otras líneas se mantuvieron por debajo de ese nivel. Esta diferencia sugiere que las condiciones operativas, configuración de maquinaria, gestión de operadores o mezcla de productos no son homogéneas entre líneas. La brecha de eficiencia representa una oportunidad de mejora importante mediante estandarización operativa y replicación de buenas prácticas de la Línea C.

2. ¿En qué turno se producen más defectos?
   El turno de **Mañana** presentó la mayor tasa promedio de defectos (**6.18%**), lo que resulta relevante porque normalmente este turno concentra mayor supervisión y disponibilidad de recursos. Esto puede indicar problemas en la etapa de arranque de producción: calibraciones iniciales deficientes, variaciones térmicas al inicio de operación, ajustes frecuentes de maquinaria o inconsistencias en preparación de materiales. El hallazgo sugiere que una parte importante de los defectos podría originarse en procesos de setup y estabilización temprana.

3. ¿Qué máquina genera más tiempos de paro?
   La máquina **Torno-02** acumuló más de **2115 minutos de paro**, convirtiéndose en el principal foco de ineficiencia operacional. Considerando el volumen total de producción, este nivel de indisponibilidad probablemente impacta directamente la capacidad instalada, el cumplimiento de órdenes y el costo operativo por unidad. Además, el hecho de que la causa de paro más frecuente sea “Sin causa” evidencia debilidad en los procesos de captura de información operativa, limitando la capacidad de análisis raíz y toma de decisiones basada en datos.

4. ¿Cómo ha evolucionado la producción semana a semana?
   La producción semanal mostró un comportamiento variable a lo largo del año, con semanas de alta capacidad productiva seguidas de caídas importantes. La **semana 43** registró el mayor volumen con **28,856 unidades**, lo que demuestra que la planta sí tiene capacidad para operar a niveles altos cuando las condiciones son favorables. Sin embargo, la volatilidad observada podría indicar sensibilidad a fallas operativas, disponibilidad de maquinaria, programación de producción o demanda. Esto sugiere que la planta aún no opera bajo un nivel de estabilidad suficientemente controlado.

5. ¿Cuál es la relación entre temperatura y tasa de defectos?
   La correlación entre temperatura y defectos fue positiva pero débil (**0.12**), indicando que la temperatura no es actualmente el principal determinante de la calidad del producto. Sin embargo, la dispersión observada en el scatter plot evidencia que ciertos puntos de operación con temperaturas elevadas sí presentan picos de defectos. Esto podría indicar que la temperatura actúa como variable amplificadora en combinación con otros factores como velocidad de máquina, desgaste de herramientas o experiencia del operador.

---

1. **Insight 1:**
   La planta presenta un desempeño operativo aceptable, pero con señales claras de falta de estabilidad en los procesos. Las diferencias entre líneas, la concentración de paros en ciertas máquinas y la variabilidad semanal indican que la operación depende más de condiciones específicas que de un sistema estandarizado y controlado.

2. **Insight 2:**
   Existe una debilidad importante en la gestión de datos operacionales. El hecho de que “Sin causa” sea la categoría de paro más frecuente limita la capacidad de la empresa para implementar mantenimiento predictivo, análisis de causa raíz y mejora continua basada en evidencia.

3. **Recomendación para el gerente:**
   Se recomienda priorizar tres frentes estratégicos:

* Implementar un plan de mantenimiento preventivo y monitoreo continuo sobre Torno-02.
* Estandarizar los procedimientos de arranque del turno mañana para reducir defectos iniciales.
* Fortalecer el sistema de captura de datos operacionales obligando el registro detallado de causas de paro y variables de proceso, con el fin de evolucionar hacia una operación basada en analítica y control estadístico.


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 10 — App Streamlit
# Crea el archivo caso3_produccion_app.py en la misma carpeta
# con toda la app completa según los requisitos de la tarea
# ══════════════════════════════════════════════════════════════

# Puedes usar esta celda para escribir el código de la app
# y luego copiarlo al archivo .py

app_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px

# Tu app aquí:

'''

# Guardar la app en un archivo
with open('caso3_produccion_app.py', 'w') as f:
    f.write(app_code)

print('✅ Archivo caso3_produccion_app.py creado')
print('Ejecuta: streamlit run caso3_produccion_app.py')

✅ Archivo caso3_produccion_app.py creado
Ejecuta: streamlit run caso3_produccion_app.py
